Code file to compute the model selection criteria (AICc, BIC) for the different models. The fits are executed again in this file (instead of taking the results from the fitting files), for simplicity in code writing. It would be best if we could directly take the fitting results and only calculate the statistics from there.

## Model selection for tumour growth only models (Table S1)

In [ ]:
using CSV, DataFrames, DifferentialEquations, Statistics
using DifferentialEquations, LsqFit

include("../src/Model_analysis_aic_bic.jl")
include("../src/Predictions.jl")
include("../src/Data.jl")
include("../src/Fixed_params.jl")
using .Predictions

In [ ]:
data = DataTools.load_tumor_datasets(data_dir = joinpath(@__DIR__, "..", "Data"))
tdata_mp = data.mp.t
ydata_mp = data.mp.y
tdata_mpb1 = data.mpb1.t
ydata_mpb1 = data.mpb1.y
u0_mp = data.mp.u0
u0_mpb1 = data.mpb1.u0

tumour_model_exponential_mp = (t, p) -> Predictions.predict_exponential(t, p, u0_mp)
tumour_model_exponential_mpb1 = (t, p) -> Predictions.predict_exponential(t, p, u0_mpb1)
tumour_model_logistic_mp = (t, p) -> Predictions.predict_logistic(t, p, u0_mp)
tumour_model_logistic_mpb1 = (t, p) -> Predictions.predict_logistic(t, p, u0_mpb1)
tumour_model_gompertz_mp = (t, p) -> Predictions.predict_gompertz(t, p, u0_mp)
tumour_model_gompertz_mpb1 = (t, p) -> Predictions.predict_gompertz(t, p, u0_mpb1)

# Initial estimates for [r, k]
r0 = 0.514  # (1/days), Tumor growth rate from de Pillis
k0 = 2000.0  # (mm^3), Tumor carrying capacity (could be more fine-tuned)

rmin = 0.0
rmax = 1.0
kmin = 0.0
kmax = 20000.0

# Fitting the models to vehicle data for MP and MPB1
fit_exponential_mp = curve_fit(tumour_model_exponential_mp, tdata_mp, ydata_mp, [r0], lower=[rmin], upper=[rmax])
fit_exponential_mpb1 = curve_fit(tumour_model_exponential_mpb1, tdata_mpb1, ydata_mpb1, [r0], lower=[rmin], upper=[rmax])
fit_logistic_mp = curve_fit(tumour_model_logistic_mp, tdata_mp, ydata_mp, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax])
fit_logistic_mpb1 = curve_fit(tumour_model_logistic_mpb1, tdata_mpb1, ydata_mpb1, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax])
fit_gompertz_mp = curve_fit(tumour_model_gompertz_mp, tdata_mp, ydata_mp, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax])
fit_gompertz_mpb1 = curve_fit(tumour_model_gompertz_mpb1, tdata_mpb1, ydata_mpb1, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax]);

In [ ]:
mp_fits = [fit_exponential_mp, fit_logistic_mp, fit_gompertz_mp];
mpb1_fits = [fit_exponential_mpb1, fit_logistic_mpb1, fit_gompertz_mpb1];
models = ["Exponential", "Logistic", "Gompertz"]; # Needs to match the order of the _fits vector above
conds = ["MP","MPB1"];

model_selection_results = compute_statistics(
    mp_fits,
    mpb1_fits,
    models,
    conds
)

output_dir = "model_selection_results"
isdir(output_dir) || mkpath(output_dir)

println("Model selection results:")
for (cond, df) in model_selection_results
    println("\nCondition: $cond")
    println(df)
    CSV.write(joinpath(output_dir, "model_selection_tumour_growth_$(cond).csv"), df)
end

## Logistic growth for ICB data and extended immune model for ICB data (Table S2)

In [ ]:
# Tumour growth with logistic model only, for ICB data

# Load tumour volume data
data = DataTools.load_tumor_datasets(data_dir = joinpath(@__DIR__, "..", "Data"))
tdata_mp_icb = data.mp_icb.t;
ydata_mp_icb = data.mp_icb.y;
tdata_mpb1_icb = data.mpb1_icb.t;
ydata_mpb1_icb = data.mpb1_icb.y;

u0_mp_icb = data.mp_icb.u0;
u0_mpb1_icb = data.mpb1_icb.u0;

tumour_model_logistic_mp_icb = (t, p) -> Predictions.predict_logistic(t, p, u0_mp_icb)
tumour_model_logistic_mpb1_icb = (t, p) -> Predictions.predict_logistic(t, p, u0_mpb1_icb)

# Initial estimates for [r, k]
r0 = 0.514  # (1/days), Tumor growth rate from de Pillis
k0 = 2000.0  # (mm^3), Tumor carrying capacity (could be more fine-tuned)
rmin = 0.0
rmax = 1.0
kmin = 0.0
kmax = 20000.0

fit_logistic_mp_icb = curve_fit(tumour_model_logistic_mp_icb, tdata_mp_icb, ydata_mp_icb, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax]);
fit_logistic_mpb1_icb = curve_fit(tumour_model_logistic_mpb1_icb, tdata_mpb1_icb, ydata_mpb1_icb, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax]);

In [ ]:
# Tumour and immune cells model for ICB (only fitting to tumour volumes)

# Definition of the model fixed parameters
fixed_params_mp = fixed_params_mp_icb;
fixed_params_mpb1 = fixed_params_mpb1_icb;

# Load fitted parameters from timeshift fitting (will be initial estimates for fitting the immune model)
time_shift_result = CSV.read(joinpath(@__DIR__, "..", "Fitted_params_results/fitted_parameters_normalised_timeshift.csv"), DataFrame)

# Case where we initialise the tumour at treatment initiation 
# Initial conditions for immune cells and tumour growth model (vehicle)
# Where T, N, E = u
# with T in mm3, N and E in cells
N0_treatment_initiation_mp = time_shift_result[time_shift_result.Dataset .== "MP", :nk_cells_at_treatment_initiation][1]
N0_treatment_initiation_mpb1 = time_shift_result[time_shift_result.Dataset .== "MPB1", :nk_cells_at_treatment_initiation][1]
E0_treatment_initiation_mp = time_shift_result[time_shift_result.Dataset .== "MP", :t_cells_at_shift_at_treatment_initiation][1]
E0_treatment_initiation_mpb1 = time_shift_result[time_shift_result.Dataset .== "MPB1", :t_cells_at_shift_at_treatment_initiation][1]

u0_immune_icb_mp = [ydata_mp_icb[1], N0_treatment_initiation_mp, E0_treatment_initiation_mp];
u0_immune_icb_mpb1 = [ydata_mpb1_icb[1], N0_treatment_initiation_mpb1, E0_treatment_initiation_mpb1];

# Initial estimates for fitted params
q_0_mp = 2.01e-3    # (1/mm3*days), from prior fitting of tumour-immune cells model to vehicle
q_0_mpb1 = 7.76e-4  # (1/mm3*days), from prior fitting of tumour-immune cells model to vehicle

# Bounds
q_min = 0.00001;
q_max = 1.0;

tumour_immune_model_mp_with_immune_icb = (t, p) -> begin
    states = Predictions.predict_tumour_immune_icb_q_states(t, p, u0_immune_icb_mp, fixed_params_mp_icb)
    return states[1, :]  # Only return the tumour volume predictions, as fitting is done to tumour volume data only
end
tumour_immune_model_mpb1_with_immune_icb = (t, p) -> begin
    states = Predictions.predict_tumour_immune_icb_q_states(t, p, u0_immune_icb_mpb1, fixed_params_mpb1_icb)
    return states[1, :]  # Only return the tumour volume predictions, as fitting is done to tumour volume data only
end

fit_immune_icb_mp = curve_fit(tumour_immune_model_mp_with_immune_icb, tdata_mp_icb, ydata_mp_icb, [q_0_mp], lower=[q_min], upper=[q_max]);
fit_immune_icb_mpb1 = curve_fit(tumour_immune_model_mpb1_with_immune_icb, tdata_mpb1_icb, ydata_mpb1_icb, [q_0_mpb1], lower=[q_min], upper=[q_max]);

In [ ]:
# Comparison

mp_fits = [fit_logistic_mp_icb, fit_immune_icb_mp];
mpb1_fits = [fit_logistic_mpb1_icb, fit_immune_icb_mpb1];
models = ["Logistic", "Immune cells"]; # Needs to match the order of the _fits vector above
conds = ["MP","MPB1"];

model_selection_results = compute_statistics(
    mp_fits,
    mpb1_fits,
    models,
    conds
)

output_dir = "model_selection_results"
isdir(output_dir) || mkpath(output_dir)

println("Model selection results:")
for (cond, df) in model_selection_results
    println("\nCondition: $cond")
    println(df)
    CSV.write(joinpath(output_dir, "model_selection_normalised_icb_$(cond).csv"), df)
end

## Comparing logistic Cis+ICB and tumour-immune cis PK/PD + ICB (no refit) (Table S3)

In [ ]:
# Logistic growth with cis+icb data

data = DataTools.load_tumor_datasets(data_dir = joinpath(@__DIR__, "..", "Data"))
tdata_mp_cis_icb = data.mp_cis_icb.t
ydata_mp_cis_icb = data.mp_cis_icb.y
tdata_mpb1_cis_icb = data.mpb1_cis_icb.t
ydata_mpb1_cis_icb = data.mpb1_cis_icb.y
u0_mp_cis_icb = data.mp_cis_icb.u0
u0_mpb1_cis_icb = data.mpb1_cis_icb.u0

tumour_model_logistic_mp_cis_icb = (t, p) -> Predictions.predict_logistic(t, p, u0_mp_cis_icb)
tumour_model_logistic_mpb1_cis_icb = (t, p) -> Predictions.predict_logistic(t, p, u0_mpb1_cis_icb)

# Initial estimates for [r, k]
r0 = 0.514  # (1/days), Tumor growth rate from de Pillis
k0 = 2000.0  # (mm^3), Tumor carrying capacity (could be more fine-tuned)
rmin = 0.0
rmax = 1.0
kmin = 0.0
kmax = 20000.0

fit_logistic_mp_cis_icb = curve_fit(tumour_model_logistic_mp_cis_icb, tdata_mp_cis_icb, ydata_mp_cis_icb, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax]);
fit_logistic_mpb1_cis_icb = curve_fit(tumour_model_logistic_mpb1_cis_icb, tdata_mpb1_cis_icb, ydata_mpb1_cis_icb, [r0, k0], lower=[rmin, kmin], upper=[rmax, kmax]);


mp_fits = [fit_logistic_mp_cis_icb];
mpb1_fits = [fit_logistic_mpb1_cis_icb];
models = ["Logistic"]; # Needs to match the order of the _fits vector above
conds = ["MP","MPB1"];

model_selection_results = compute_statistics(
    mp_fits,
    mpb1_fits,
    models,
    conds
)

output_dir = "model_selection_results"
isdir(output_dir) || mkpath(output_dir)

println("Model selection results:")
for (cond, df) in model_selection_results
    println("\nCondition: $cond")
    println(df)
    CSV.write(joinpath(output_dir, "model_selection_normalised_cis_icb_$(cond).csv"), df)
end

In [ ]:
# Manually calculating AICc and BIC for model with no refit

data = DataTools.load_tumor_datasets(data_dir = joinpath(@__DIR__, "..", "Data"))
tdata_mp_cis_icb = data.mp_cis_icb.t
ydata_mp_cis_icb = data.mp_cis_icb.y
tdata_mpb1_cis_icb = data.mpb1_cis_icb.t
ydata_mpb1_cis_icb = data.mpb1_cis_icb.y
u0_mp_cis_icb = data.mp_cis_icb.u0
u0_mpb1_cis_icb = data.mpb1_cis_icb.u0

# Initial conditions for immune cells:
# Load fitted parameters from timeshift fitting (will be initial estimates for fitting the immune model)
time_shift_result = CSV.read(joinpath(@__DIR__, "..", "Fitted_params_results/fitted_parameters_normalised_timeshift.csv"), DataFrame)
N0_treatment_initiation_mp = time_shift_result[time_shift_result.Dataset .== "MP", :nk_cells_at_treatment_initiation][1]
N0_treatment_initiation_mpb1 = time_shift_result[time_shift_result.Dataset .== "MPB1", :nk_cells_at_treatment_initiation][1]
E0_treatment_initiation_mp = time_shift_result[time_shift_result.Dataset .== "MP", :t_cells_at_shift_at_treatment_initiation][1]
E0_treatment_initiation_mpb1 = time_shift_result[time_shift_result.Dataset .== "MPB1", :t_cells_at_shift_at_treatment_initiation][1]

# Load fitted parameters
param_immune_cis = CSV.read("../Fitted_params_results/fitted_parameters_normalised_immune_cisplatin.csv", DataFrame)
# Extract immune model parameters for MP and MPB1
#k2_immune_mp = param_immune_cis[param_immune_cis.Dataset .== "MP", :k2][1]
k2N_immune_mp = param_immune_cis[param_immune_cis.Dataset .== "MP", :k2N][1]
k2E_immune_mp = param_immune_cis[param_immune_cis.Dataset .== "MP", :k2E][1]
#k2_immune_mpb1 = param_immune_cis[param_immune_cis.Dataset .== "MPB1", :k2][1]
k2N_immune_mpb1 = param_immune_cis[param_immune_cis.Dataset .== "MPB1", :k2N][1]
k2E_immune_mpb1 = param_immune_cis[param_immune_cis.Dataset .== "MPB1", :k2E][1]
# Load fitted parameters
param_immune_icb = CSV.read("../Fitted_params_results/fitted_parameters_normalised_immune_icb.csv", DataFrame)
# Extract immune model parameters for MP and MPB1
q_immune_mp = param_immune_icb[param_immune_icb.Dataset .== "MP", :q][1]
q_immune_mpb1 = param_immune_icb[param_immune_icb.Dataset .== "MPB1", :q][1]

include("../src/Fixed_params.jl")
# Definition of the model fixed parameters
fixed_params_mp_cis_icb = deepcopy(fixed_params_icb_cis_mp)
fixed_params_mpb1_cis_icb = deepcopy(fixed_params_icb_cis_mpb1)

# Fixed bolus dosing setup
BW = 0.025
dose_amt = 3.0 * BW
initial_day_dose = 0.01  # Start dosing at 0.01 days to avoid dosing at t=0 which is causing issues
dose_starts = [initial_day_dose, initial_day_dose + 7.0]  # Doses 1 week apart

# Initial conditions
u0_immune_cis_icb_mp = [0, 0, ydata_mp_cis_icb[1], 0, 0, 0, N0_treatment_initiation_mp, E0_treatment_initiation_mp];
u0_immune_cis_icb_mpb1 = [0, 0, ydata_mpb1_cis_icb[1], 0, 0, 0, N0_treatment_initiation_mpb1, E0_treatment_initiation_mpb1];

predictions_cis_icb_mp = Predictions.predict_tumour_immune_cis_pkpd_states(tdata_mp_cis_icb, [8.07, k2N_immune_mp, k2E_immune_mp], u0_immune_cis_icb_mp, fixed_params_mp_cis_icb, dose_amt, dose_starts)
predictions_cis_icb_mpb1 = Predictions.predict_tumour_immune_cis_pkpd_states(tdata_mpb1_cis_icb, [63.7,k2N_immune_mpb1, k2E_immune_mpb1], u0_immune_cis_icb_mpb1, fixed_params_mpb1_cis_icb, dose_amt, dose_starts)

# Extract T, N, E for each dataset
yfit_T_cis_icb_mp = predictions_cis_icb_mp[1, :];
yfit_T_cis_icb_mpb1 = predictions_cis_icb_mpb1[1, :];


## Residuals and model selection criteria for no refit case
nofit_residuals_mp = ydata_mp_cis_icb .- yfit_T_cis_icb_mp
nofit_residuals_mpb1 = ydata_mpb1_cis_icb .- yfit_T_cis_icb_mpb1
rss_mp = sum(nofit_residuals_mp.^2)
rss_mpb1 = sum(nofit_residuals_mpb1.^2)
n_mp = length(ydata_mp_cis_icb)
k_mp = 0   # no fitted parameters
n_mpb1 = length(ydata_mpb1_cis_icb)
k_mpb1 = 0   # no fitted parameters

AICc_mp_nofit = AICc(rss_mp, k_mp, n_mp)
AICc_mpb1_nofit = AICc(rss_mpb1, k_mpb1, n_mpb1)
BIC_mp_nofit = BIC(rss_mp, k_mp, n_mp)
BIC_mpb1_nofit = BIC(rss_mpb1, k_mpb1, n_mpb1)

println("Model selection results for no refit case:")
println("\nCondition: MP")
println("AICc: $AICc_mp_nofit")
println("BIC: $BIC_mp_nofit")
println("RSS: $rss_mp")
println("\nCondition: MPB1")
println("AICc: $AICc_mpb1_nofit")
println("BIC: $BIC_mpb1_nofit")
println("RSS: $rss_mpb1")